# Assignment 02: Project AeroAssist

**Course:** Agentic AI: From Concepts to Practice

**Instructor:** Prof. Karthik Vaidhyanathan

**TAs:** Aviral Gupta, Aneetta Sara Shany, Ch Pavan Harshit, Shreyash Chandak

---

## Student Details

**Name:** Kapil Kumar

**Roll Number:**  evernorth-aai-1197201

**Email:**  kapil.mnnit90@gmail.com

---

## Instructions

- Run the notebook from top to bottom.
- Complete all coding tasks.
- Answer every reflection question in markdown.
- Do **not** hardcode your Gemini API key.
- Submit only the completed notebook.

# Part 1 — Building the Chatbot

## Objective

Create a Flask application that serves a simple chatbot interface.

By the end of this section you should have a working webpage with a message
box and a send button that talks to a Flask backend.

In [1]:
# Install required packages
# Run this cell first — it may take a minute

!pip install flask flask-cors google-generativeai

In [2]:
# Imports

import threading
import time
from flask import Flask, request, jsonify
from flask_cors import CORS
from google.colab.output import eval_js
import google.generativeai as genai

print('All libraries imported successfully.')

All libraries imported successfully.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# Read your Gemini API key from Colab Secrets
#
# Steps:
#   1. Click the key icon in the left sidebar
#   2. Add a secret named GEMINI_API_KEY and paste your key as the value
#   3. Toggle 'Notebook access' ON

from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
print('API key loaded.' if GEMINI_API_KEY else 'WARNING: API key not found.')

API key loaded.


In [4]:
# Create the Flask application

app = Flask(__name__)
CORS(app)  # Allows the browser page to send requests to our backend

print('Flask app created.')

Flask app created.


In [5]:
# HTML for the chatbot interface
#
# This is a minimal starting point. The weird UI is deliberately made for you to fix.
# The three elements that matter are:
#   <div id='chat-window'>  — where messages will be displayed
#   <input id='user-input'> — where the user types
#   <button id='send-btn'>  — sends the message
#
# TODO: Build your HTML page below.
#       Add a <style> block inside <head> if you want to improve the look.
#       Hints: try background colours, font-family, padding, border-radius.

# HTML for the chatbot interface
#
# Part 1: Improve the chatbot UI

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
    <title>AeroAssist Chatbot</title>

    <style>
        body {
            font-family: Arial, sans-serif;
            background-color: #f4f7fb;
            display: flex;
            justify-content: center;
            align-items: flex-start;
            min-height: 100vh;
            margin: 0;
            padding-top: 40px;
        }

        #chat-window {
            background-color: white;
            width: 450px;
            padding: 20px;
            border-radius: 12px;
            box-shadow: 0 4px 15px rgba(0, 0, 0, 0.15);
        }

        h3 {
            text-align: center;
            margin-top: 0;
            color: #333;
        }

        #messages {
            height: 300px;
            overflow-y: auto;
            border: 1px solid #ddd;
            border-radius: 8px;
            padding: 10px;
            margin-bottom: 12px;
            background-color: #fafafa;
        }

        #user-input {
            width: 70%;
            padding: 10px;
            border: 1px solid #ccc;
            border-radius: 6px;
            box-sizing: border-box;
        }

        #send-btn {
            width: 25%;
            padding: 10px;
            margin-left: 5px;
            background-color: #007bff;
            color: white;
            border: none;
            border-radius: 6px;
            cursor: pointer;
        }

        #send-btn:hover {
            background-color: #0056b3;
        }
    </style>
</head>

<body>

    <div id="chat-window">
        <h3>AeroAssist Chatbot</h3>

        <div id="messages"></div>

        <input
            id="user-input"
            type="text"
            placeholder="Type your message..."
        />

        <button id="send-btn" onclick="sendMessage()">
            Send
        </button>
    </div>

"""

print('HTML template defined.')

HTML template defined.


In [6]:
# JavaScript that connects the frontend to the Flask backend
#
# The script below gives you the structure — fill in the two TODOs.
#
# How it works:
#   1. The user clicks Send (or presses Enter)
#   2. sendMessage() reads the text from #user-input
#   3. It sends a POST request to /chat with the message as JSON
#   4. The server replies with JSON: { response: '...', model: '...' }
#   5. addMessage() creates a new element and appends it to #chat-window

# JavaScript that connects the frontend to the Flask backend

JS_SCRIPT = """
<script>

function addMessage(sender, text) {
    const messages = document.getElementById("messages");

    const messageDiv = document.createElement("div");
    messageDiv.innerHTML = `<b>${sender}:</b> ${text}`;

    messages.appendChild(messageDiv);
    messages.scrollTop = messages.scrollHeight;
}

function sendMessage() {

    const inputBox = document.getElementById("user-input");
    const message = inputBox.value.trim();

    if (!message) return;

    addMessage("You", message);

    inputBox.value = "";

    fetch("/chat", {
        method: "POST",
        headers: {
            "Content-Type": "application/json"
        },
        body: JSON.stringify({
            message: message
        })
    })
    .then(response => response.json())
    .then(data => {
        addMessage("Bot", data.response);
    })
    .catch(error => {
        console.error("Error:", error);
        addMessage("Bot", "Sorry, something went wrong.");
    });
}

// Allow the user to press Enter to send the message
document.getElementById("user-input").addEventListener("keydown", function(event) {
    if (event.key === "Enter") {
        sendMessage();
    }
});

</script>
</body>
</html>
"""

FULL_PAGE = HTML_PAGE + JS_SCRIPT

print("Page assembled.")

Page assembled.


In [41]:
url = eval_js('google.colab.kernel.proxyPort(5000)')
print('AeroAssist is running at:', url)
print('Open the link above in a new tab to test your chatbot.')

#
# The frontend reads data.response and displays it in the chat window.
# You do not need to modify this route — just replace the reply line in Part 2.

# ACTIVE_MODEL = 'placeholder'

AeroAssist is running at: https://5000-m-s-kkb-use1c2-klw0f6oqpotl-c.us-east1-2.prod.colab.dev
Open the link above in a new tab to test your chatbot.


In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

# Flask routes
#
# How the /chat endpoint works:
#
#   1. The browser sends:  POST /chat  with body { "message": "..." }
#   2. Flask reads the message from the request JSON
#   3. It generates a reply (placeholder for now; replaced in Part 2)
#   4. It returns:         { "response": "...", "model": "..." }

@app.route("/")
def home():
    return FULL_PAGE

@app.route('/chat', methods=['POST'])
def chat():
    message = request.json.get("message", "").lower()
    askGemini(message)
    # Part 1: fixed reply so you can verify the frontend is working
    # TODO (Part 2): replace this with a call to askGemini(message)
    reply = f"[placeholder] You said: {message}"

    return jsonify({'response': reply, 'model': ACTIVE_MODEL})

print("static endpoints updated.")


static endpoints updated.


In [44]:
# Running flask in seperate cell.
app.run(
      host="0.0.0.0",
      port=5000,
      debug=False,
      use_reloader=False
  )

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Open the URL printed above in a new browser tab. Type a message and click Send. If you see the placeholder response appear in the chat window, Part 1 is complete.

> **Before moving on:** take a screenshot of the chatbot. Attach it in your submission zip.

**basic_chatbot.jpg attached in zip.**

# Part 2 — Connecting Gemini

## Objective

Replace the fixed placeholder reply with responses generated by Google's Gemini model.

By the end of this section your chatbot should answer real questions.

In [13]:
# Configure the Gemini client

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('gemini-3.1-flash-lite')

print('Gemini client configured.')

Gemini client configured.


In [8]:
# TODO: Complete the askGemini function.
#
# The function should:
#   - Accept a user prompt, an optional system prompt, and a temperature value
#   - Send the prompt to Gemini and return the response text
#
# The key call is:
  # response = gemini_model.generate_content(prompt, generation_config=config)
  # return response.text
#
# GenerationConfig controls temperature:
#   config = genai.types.GenerationConfig(temperature=temperature)
#
# If a system prompt is provided, prepend it to the user prompt before sending.

def askGemini(prompt, system=None, temperature=0.3):
    try:
        # Combine system instruction and user prompt
        if system:
            full_prompt = f"{system}\n\nUser: {prompt}"
        else:
            full_prompt = prompt

        # Generate response from Gemini
        response = gemini_model.generate_content(
            full_prompt,
            generation_config={
                "temperature": temperature
            }
        )

        return response.text

    except Exception as e:
        return f"Error generating response: {str(e)}"


# Test it
print(askGemini('What is the capital of France?'))

The capital of France is Paris.


In [ ]:
# Update the /chat endpoint to use Gemini.
#
# Replace the placeholder reply with a call to askGemini(message).
# Re-run the cell below to relaunch the server with this change.

ACTIVE_MODEL = 'Gemini'

app.view_functions.pop('chat', None) # This is needed to remove the previously defined route. Use this wherever you redefine.
@app.route('/chat', methods=['POST'])
def chat():
    message = request.json.get('message', '')
    reply = askGemini(message)
    reply = f"[placeholder] You said: {message}"
    return jsonify({'response': reply, 'model': ACTIVE_MODEL})

print("genai endpoint updated")


genai endpoint updated


## Experiment

**Ask your chatbot at least five different questions — a mix of general knowledge,
airline questions, and aircraft specifications.**

**Note anything interesting: does Gemini always answer the same way? Does it ever
make up airline-specific details it could not possibly know?**


I tested the chatbot with multiple questions about general knowledge, airlines and aircraft specifications. Gemini gave mostly relevant answers and the answers were understandable. When I asked the same question more than once, the answer was not exactly same every time, but it was mostly similar. For general knowledge and aircraft specifications, the answers looked mostly consistent. So, I think this chatbot is useful for general information, but for more specific information it's better to verify from the official sources.


# Part 3 — Understanding Your AI Assistant

## Objective

Investigate how different language models behave and how the way you instruct
a model shapes its responses.

This section is about experimentation — there are no single correct answers.

In [74]:
import subprocess
import time

# Step 0: Install zstd, a compression utility required by Ollama
print('Installing zstd...')
!sudo apt-get install -y zstd

# Step 1: Download and install Ollama
print('Installing Ollama...')
!curl -fsSL https://ollama.com/install.sh | sh

# Step 2: Start the Ollama server in the background
print('Starting Ollama server...')
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Wait for the server to be ready

# Step 3: Download a small language model
# gemma3:1b is a compact model from Google — about 815 MB. You are free to experiment with other models as well.
# This may take 2-4 minutes depending on your connection
print('Downloading gemma3:1b — this may take a few minutes...')
!ollama pull gemma3:1b

print('Done. Ollama is ready.')

Installing zstd...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Installing Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama server...

Done. Ollama is ready.


In [75]:
# TODO: Complete the askLocalModel function.
#
# Ollama exposes a local HTTP API at http://localhost:11434/api/generate
# Send a POST request with a JSON body and read the response.
#
# Required fields in the request body:
#   'model'  : the model name, e.g. 'gemma3:1b'
#   'prompt' : the user's message
#   'stream' : False  (so we get the full reply at once)
#
# Optional field:
#   'system' : a system prompt (same idea as in askGemini)
#
# The response JSON contains a 'response' key with the generated text.
#
# Hint: use the 'requests' library (import it as http_requests to avoid
#       clashing with Flask's 'request' object)

import requests as http_requests

OLLAMA_MODEL = 'gemma3:1b'

def askLocalModel(prompt, system=None, model=OLLAMA_MODEL, temperature=None):
    url = 'http://localhost:11434/api/generate'

    # Required request body
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': False
    }

    # Add optional system prompt
    if system is not None:
        payload['system'] = system

    # Add temperature if provided
    if temperature is not None:
        payload['options'] = {
            'temperature': temperature
        }

    # Send request to Ollama
    response = http_requests.post(
        url,
        json=payload
    )

    # Raise an error if the HTTP request failed
    response.raise_for_status()

    # Read response JSON
    data = response.json()

    # Return generated text
    return data['response']




In [76]:
# Test Olama
print(askLocalModel('What is the capital of France?'))

Paris! 🌱 😊 

It’s the definitive capital of Francia. 🇫🇷👍‍


## Experiment 1 — Gemini vs Ollama

Use the five questions below to compare both models.
For each question, call both `askGemini` and `askLocalModel` and print the results side by side.

In [17]:
# gemini_response = askGemini("What is the ?baggage allowance for economy class?")
# print(f"gemini_response: {gemini_response}")

ollama_response = askLocalModel("What is the baggage allowance for economy class?")
print(f"ollama_response: {ollama_response}")

ollama_response: Okay, let's break down baggage allowances in economics classes. It’s a surprisingly complex area with different regulations depending on where you are traveling (the specific country and airline). Here’s what we need to know:

**Generally, Economy Class Baggage is Limited:**

* **Maximum Allowance:** In most cases, *no more than 2% of your total checked baggage weight*. This is the key takeaway. That means if your entire suitcase weighs 5 kilograms (11 pounds), you’ll generally only be allowed to bring 1 kg (2lbs).
*   **The exact amount varies greatly by country:**
    * **Europe (Mostly):**  Typically, you have a maximum of around **23 x 0.4kg (8-10 lbs)** for the entire luggage combined.
        * **France:** Around 60x8kg or 48 Kg/ 53lbs. 9kg max per piece. Also includes check-in bags which weigh up to 3 kg. 
         *  **Germany**: Approximately 23kg(53lbs), though the limit can be reduced to 16kg/ 32lb on some airlines. It is commonly advised to get lighter, sof

In [18]:
# Five airline customer support questions
# You can swap any of these for your own if you prefer.

questions = [
    'What is the baggage allowance for economy class?',
    'Can I change my booking after purchasing my ticket?',
    'Does the Airbus A320 have Wi-Fi?',
    'How early should I arrive at the airport before my flight?',
    'What happens if my flight is cancelled?',
]

# TODO: Loop through the questions list.
#       For each question, call askGemini and askLocalModel.
#       Print the question and both responses clearly.

# Experiment 1 — Gemini vs Ollama

# Compare Gemini and Ollama responses
for i, question in enumerate(questions, 1):
    print("=" * 80)
    print(f"Question {i}: {question}")
    print("-" * 80)

    # Get response from Gemini
    gemini_response = askGemini(question)

    print("\nGemini:")
    print(gemini_response)

    # Get response from Ollama
    ollama_response = askLocalModel(question)

    print("\nOllama:")
    print(ollama_response)

    print()


Question 1: What is the baggage allowance for economy class?
--------------------------------------------------------------------------------

Gemini:
Baggage allowance varies significantly depending on the **airline**, your **destination**, and your **fare type** (e.g., Basic Economy vs. Standard Economy).

To give you an accurate answer, I need to know:
1. **Which airline are you flying with?**
2. **What is your route?** (e.g., Domestic within the US, Transatlantic, or within Asia?)

---

### General Rules of Thumb
While policies vary, here is how most airlines handle economy baggage:

#### 1. Carry-on Baggage
*   **Standard Economy:** Most airlines allow **one carry-on bag** (to go in the overhead bin) and **one personal item** (a purse, laptop bag, or backpack that fits under the seat).
*   **Basic Economy:** Some budget airlines (like United, JetBlue, or Spirit) **do not allow a full-sized carry-on bag** for Basic Economy passengers; you are only allowed one personal item.

#### 2

After running the comparison above, answer the following in a few sentences.

**- How did the two models differ in speed and level of detail?**

Gemini answers were more detailed and faster but ollama was running in local venv so it was very slow.

**-Which gave more accurate or useful answers to the airline questions?**

I noticed Gemini's answers more accurate and useful for the airline questions. It generally gave more comprehensive answers with more context, while Ollama's answers were shorter.


**- Which would you rather put in front of a customer, and why?**

I would put Gemini in front of a customer as it's answers were more detailed and useful, and it gave better context for the airline questions compare to ollama.


## Experiment 2 — Temperature

Temperature controls how varied a model's output is.

- **0** — the model always picks the most likely next word (consistent, predictable)
- **0.5** — a balance between consistency and variety
- **1.0** — the model takes more creative risks (varied, sometimes unexpected)

Run the same prompt with all three values and compare the outputs.

In [ ]:
# TODO: Choose a customer support question and generate three responses
#       using temperatures 0, 0.5, and 1.0.
#       Print each temperature value and the corresponding response.

test_prompt = 'What should I do if I miss my flight?'

temperatures = [0, 0.5, 1.0]

for temp in temperatures:
    print("=" * 60)
    print(f"Temperature: {temp}")
    print("-" * 60)

    response = askLocalModel(
        test_prompt,
        temperature=temp
    )

    print(response)
    print()

Temperature: 0
------------------------------------------------------------
Okay, missing a flight is incredibly stressful! Here's a breakdown of what you need to do, broken down into immediate actions and longer-term considerations. **Time is your friend – act quickly.**

**1. Immediate Actions (Within the First Few Minutes - Critical!)**

*   **Check Your Airline’s App/Website:** This is *the most important thing*. Airlines usually have a dedicated section for missed flights with real-time updates, rebooking options, and compensation information.
* **Contact Customer Service Immediately:** Don't delay! Call the airline's customer service number (usually on their website or app). Be prepared to provide your booking reference/confirmation number.  Speak calmly and clearly.
*   **Report the Missed Flight:** Let them know you’ve missed your flight. They will likely ask for your details. 

**2. What to Do Next – Options & Steps**

Here's a breakdown of what you can do, depending on the ai

## Experiment 3 — Prompting Strategies

The way you phrase a request to a language model has a large effect on the quality
of its response. In this experiment you will try three prompting strategies on the
same customer support task and compare the results.

Choose one task for all three sub-experiments. Good choices include:
- Classify a customer request (e.g. complaint / refund request / general enquiry)
- Summarise a customer complaint in one sentence
- Rewrite a draft reply in a professional airline tone


### (a) Zero-shot Prompting

Ask the model to perform the task without providing any examples.


In [20]:
# Zero-shot: no examples, just the instruction and the input.

task_input = (
    "Customer message: 'I booked the wrong date and need to change my flight. "
    "The website is not letting me do it online. This is very frustrating.'"
)

zero_shot_prompt = f"""
Classify the customer request below into one of these categories:
Complaint, Booking Change, Refund Request, General Enquiry.
Then write a one-sentence professional reply.

{task_input}
"""

print('--- Zero-shot ---')
print(askGemini(zero_shot_prompt))


--- Zero-shot ---
**Category:** Booking Change

**Reply:** I sincerely apologize for the frustration caused by our website, and I would be happy to assist you in manually updating your flight to the correct date.


### (b) One-shot Prompting

Repeat the same task. This time, provide exactly one worked example before asking
your question.


In [21]:
# One-shot: one example before the real task.

one_shot_prompt = f"""
Classify the customer request and write a one-sentence professional reply.

Example:
Customer message: 'My luggage was damaged during the flight.'
Category: Complaint
Reply: We sincerely apologise for the damage to your luggage and will arrange a claim review within 24 hours.

Now classify this:
{task_input}
Category:
Reply:
"""

print('--- One-shot ---')
print(askGemini(one_shot_prompt))


--- One-shot ---
Category: Booking Modification / Technical Issue

Reply: We sincerely apologize for the frustration caused by our website and have initiated the process to update your flight to the correct date.


### (c) Few-shot Prompting

Repeat the experiment once more. Provide at least three examples before asking the
model to complete the same task.


In [22]:
# Few-shot: three examples before the real task.

few_shot_prompt = f"""
Classify the customer request and write a one-sentence professional reply.

Example 1:
Customer message: 'My luggage was damaged during the flight.'
Category: Complaint
Reply: We sincerely apologise for the damage to your luggage and will arrange a claim review within 24 hours.

Example 2:
Customer message: 'Can I get a refund for my cancelled booking?'
Category: Refund Request
Reply: Yes, refunds for cancelled bookings are processed within 7 business days to your original payment method.

Example 3:
Customer message: 'Does AeroWing fly to Dubai?'
Category: General Enquiry
Reply: AeroWing currently operates flights to over 40 destinations; please check our website for the latest route map.

Now classify this:
{task_input}
Category:
Reply:
"""

print('--- Few-shot ---')
print(askGemini(few_shot_prompt))


--- Few-shot ---
Category: Booking Modification / Technical Issue

Reply: We sincerely apologize for the frustration caused by our website and would be happy to assist you in manually updating your flight date to the correct one.


Compare the three outputs above.

- Did the format of the response change as you added more examples?
- Did the accuracy or relevance of the category improve?
- What does this suggest about how many examples are worth providing?



*(Write your observations here)*

I think the format changed a little when I added more examples. In zero-shot, the answer was more free-form, but with one and three examples, the model followed the Category and Reply format better. The category was mostly correct in all three cases, but the examples helped the model give answers in a more similar and clear way. I think 1 to 3 examples are enough for this type of task. Giving too many examples may not be required.

## Experiment 4 — System Prompts

A **system prompt** is an instruction you give the model before the conversation starts.
It defines the role, tone, and focus of every response.

Four example roles are provided below. Keep the user question fixed and observe
how each role changes the response.

In [ ]:
# Four role prompts for AeroWing staff
# Read through them — each one gives the model a different identity.

system_prompts = {
    'Customer Support Executive': (
        'You are a helpful and professional customer support executive at AeroWing Airlines. '
        'Answer questions clearly and concisely. If a question is outside your knowledge, '
        'say so politely and offer to escalate. Do not make up flight numbers, prices, or policies.'
    ),
    'Aircraft Maintenance Engineer': (
        'You are an aircraft maintenance engineer at AeroWing Airlines. '
        'Answer technical questions about aircraft systems, components, and safety procedures '
        'using precise engineering language. When in doubt, recommend consulting the AMM.'
    ),
    'Flight Operations Manager': (
        'You are a flight operations manager at AeroWing Airlines. '
        'You focus on scheduling, crew management, regulatory compliance, and operational efficiency. '
        'Answer questions from an operational planning perspective.'
    ),
    'Travel Consultant': (
        'You are a friendly travel consultant helping passengers plan their journeys with AeroWing Airlines. '
        'Focus on making travel enjoyable and stress-free.'
    ),
}

user_question = 'What should I know before boarding the aircraft?'

# TODO: Loop through system_prompts and call askGemini for each one,
#       passing the role's prompt as the 'system' argument.
#       Print the role name and the response for each.

# Hint: Iterate over all (role_name, system_prompt) pairs, call the model
# with the same user_question each time, and print both the role name
# and the generated response.


# Loop through all four system prompts
for role_name, system_prompt in system_prompts.items():
    print("=" * 80)
    print(f"Role: {role_name}")
    print("-" * 80)

    response = askGemini(
        user_question,
        system=system_prompt
    )

    print(response)
    print()



Role: Customer Support Executive
--------------------------------------------------------------------------------
Hello! Thank you for reaching out to AeroWing Airlines. I would be happy to assist you with your travel preparations.

To ensure a smooth boarding process, please keep the following in mind:

*   **Documentation:** Have your boarding pass (digital or printed) and a valid government-issued photo ID ready for inspection at the gate.
*   **Carry-on Limits:** Please ensure your carry-on luggage meets our size and weight requirements. Items that exceed these limits may need to be checked at the gate.
*   **Boarding Time:** We recommend arriving at your departure gate at least 30–45 minutes before your scheduled departure time, as boarding typically concludes 15 minutes prior to takeoff.
*   **Prohibited Items:** Please review our list of restricted items to ensure your carry-on does not contain any prohibited materials.
*   **Assistance:** If you require special assistance (such

## Experiment 5 — Providing Additional Context

Gemini knows general aviation facts, but it does not know AeroWing's specific
aircraft or policies. One simple fix is to paste relevant information directly
into the prompt.

Compare responses with and without the AeroWing reference document below.

In [ ]:
# AeroWing quick-reference document
# This is the information the model does not have by default.

AEROWING_CONTEXT = """
AEROWING AIRLINES — AIRCRAFT QUICK REFERENCE

Airbus A320 (Narrow-body, Short-to-Medium Haul)
  Seating: 150-180 passengers (economy + business)
  Range: approximately 6,100 km
  Cruising speed: 840 km/h at 35,000 ft
  Wi-Fi: Available on select routes (check booking confirmation)
  Baggage: 23 kg checked + 7 kg cabin per Economy ticket
  Features: Winglets on A320neo variant for better fuel efficiency

Boeing 737-800 (Narrow-body, Short-to-Medium Haul)
  Seating: 160-190 passengers
  Range: approximately 5,765 km
  Cruising speed: 855 km/h
  Wi-Fi: Not available on this fleet
  Baggage: 23 kg checked + 7 kg cabin per Economy ticket

AeroWing Policies
  Booking changes: Allowed up to 24 hours before departure (fee may apply)
  Cancellations: Refundable within 24 hours of booking; non-refundable after
  Check-in: Online check-in opens 48 hours before departure
  Seat selection: Free for business class; fee applies for economy window/aisle
  Special meals: Request at least 48 hours before departure
"""

question = "Does AeroWing's A320 have Wi-Fi, and what is the baggage allowance?"

# TODO: Call askGemini twice with the same question:
#       1. Without any context — just pass the question as the prompt
#       2. With context — prepend AEROWING_CONTEXT to the prompt before
#          asking, and instruct the model to only use the provided information
#       Print both responses and compare.


# Without context
response_without_context = askGemini(question)

print('--- Without AeroWing Context ---')
print(response_without_context)


# With context
context_prompt = f"""
Use only the AeroWing reference information provided below to answer the question.
Do not make up or add any information that is not in the reference.

AeroWing Reference:
{AEROWING_CONTEXT}

Question:
{question}
"""

response_with_context = askGemini(context_prompt)

print('\n--- With AeroWing Context ---')
print(response_with_context)



--- Without AeroWing Context ---
To provide you with the most accurate information, it is important to note that **"AeroWing" is not a major commercial airline.** It is likely a fictional name, a small charter operator, or a regional flight school.

Because there is no global airline by that name operating A320s, here is how you can find the information you need:

### 1. Verify the Airline Name
*   **Check your booking confirmation:** Look closely at your ticket or confirmation email. Are you flying with a different airline (e.g., Aer Lingus, AeroMexico, AirAsia)? 
*   **Check the IATA code:** If you have a flight number (e.g., AW123), the two-letter code at the start will tell you exactly which airline is operating the flight. You can search that code on [IATA’s website](https://www.iata.org/en/publications/directories/code-search/).

### 2. If it is a Charter or Private Operator
If you are flying with a private charter company named AeroWing:
*   **Wi-Fi:** Most private charter A320s

## Reflection

1. **Which experiment changed the model's responses the most — temperature, system prompts,
   or additional context? Why do you think that is?**

   I think the additional context changed the answer the most. When we gave Gemini the AeroWing information, it gave a more specific answer about the airline. Temperature mostly changed the answer style, and system prompts changed the way the model answered based on its role. But giving extra information helped Gemini know the actual AeroWing policies.

2. **What would happen if AeroWing had hundreds of pages of policy documents instead
   of a short paragraph? How do you think that problem could be solved?
   (You are not expected to implement a solution.)**
   
   If AeroWing had hundreds of pages of policy documents, it would be difficult to put all of them in the prompt every time. The prompt would become too long and it may also take more time to get the answer. I think RAG could be used for this. It can search the documents and give only the useful information to the model based on the customer's question. This can help the chatbot answer questions using the company documents without adding all the documents in every prompt.


(your answers here)

# Part 4 — Running a Local Language Model

## Objective

Modify the chatbot so that it can serve responses from either Gemini or Ollama,
without changing a single line of the frontend.

In [46]:
# TODO: Write a unified ask() function that routes every request to
#       whichever model is currently selected by ACTIVE_MODEL.
#
# The function signature:
#   ask(prompt, system=None, temperature=0.3)
#
# If ACTIVE_MODEL == 'Gemini'  -> call askGemini(...)
# If ACTIVE_MODEL == 'Ollama'  -> call askLocalModel(...)
# Otherwise                    -> return '[No model selected]'
#
# The rest of the application will only ever call ask().
# This means switching models is a one-variable change.

def ask(prompt, system=None, temperature=0.3):
    if ACTIVE_MODEL == 'Gemini':
        return askGemini(
            prompt,
            system=system,
            temperature=temperature
        )

    elif ACTIVE_MODEL == 'Ollama':
        return askLocalModel(
            prompt,
            system=system,
            temperature=temperature
        )

    else:
        return '[No model selected]'


# Test with Gemini
# ACTIVE_MODEL = 'Ollama'
ACTIVE_MODEL = 'Gemini'
print(ask('Say hello in one sentence.'))

Hello, it is a pleasure to meet you!


In [25]:
# TODO: Write a system prompt for AeroAssist.
#
# This prompt will be sent to whichever model is active.
# It should tell the assistant:
#   - What airline it works for
#   - What kind of questions it should answer
#   - How it should behave when it does not know something

CHATBOT_SYSTEM_PROMPT = (
    "You are AeroAssist, a helpful customer support assistant for AeroWing Airlines. "
    "Answer questions related to AeroWing Airlines, flights, bookings, baggage, "
    "aircraft, and general air travel. "
    "Give clear, polite, and useful answers. "
    "If you do not know the answer or do not have enough information, say so honestly "
    "and do not make up airline policies, flight numbers, prices, schedules, or other details."
)

print('System prompt defined.')

System prompt defined.


In [52]:
# Model selector
#
# Change this variable to switch between Gemini and Ollama.
# Nothing else in the application needs to change.
#
# Options: 'Gemini'  or  'Ollama'

ACTIVE_MODEL = 'Ollama'   # <── change this to switch models

print(f'Active model: {ACTIVE_MODEL}')

Active model: Ollama


In [71]:
url = eval_js('google.colab.kernel.proxyPort(5000)')
print('AeroAssist is running at:', url)
print('Open the link above in a new tab to test your chatbot.')



AeroAssist is running at: https://5000-m-s-kkb-use1c2-klw0f6oqpotl-c.us-east1-2.prod.colab.dev
Open the link above in a new tab to test your chatbot.


In [72]:
app.run(
      host="0.0.0.0",
      port=5000,
      debug=False,
      use_reloader=False
  )

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [26/Jul/2026 18:26:33] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/Jul/2026 18:27:01] "POST /chat HTTP/1.1" 200 -


In [63]:
# Update the /chat route to use ask() and CHATBOT_SYSTEM_PROMPT.
# The /model route is provided — it lets the frontend show which model is active.

# Update the /chat route to use ask() and CHATBOT_SYSTEM_PROMPT.
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/")
def home():
    return FULL_PAGE

@app.route('/model')
def model_info():
    return jsonify({'model': ACTIVE_MODEL})


app.view_functions.pop('chat', None)

@app.route('/chat', methods=['POST'])
def chat():
    message = request.json.get('message', '')

    reply = ask(
        message,
        system=CHATBOT_SYSTEM_PROMPT
    )

    return jsonify({
        'response': reply,
        'model': ACTIVE_MODEL
    })


> **Before moving on:** take a screenshot of the chatbot with Gemini active, then change `ACTIVE_MODEL = 'Ollama'`, re-run the selector cell, refresh the tab, and take a second screenshot. Attach both in your submission zip.


**Gemini.jpg and Olama.jpg attached in zip.**


# Final Reflection

Answer each question in three to five sentences.
Support your answers with observations from your own experiments.

---

**1.Compare cloud-hosted and locally hosted language models. Discuss one advantage and one limitation of each**

Cloud-hosted models like Gemini are easy to use and usually give more detailed and useful answers. In my experiment, Gemini gave better answers for the airline questions compared to the local Ollama model. One limitation is that cloud models need an internet connection and may not always know specific company information. Local models like Ollama can run on your own system and give faster responses, but the answers can be shorter and less detailed.


---

**2.Suppose AeroWing plans to deploy specialised assistants for Customer Support,
Flight Operations, and Aircraft Maintenance. Would you use the same system
prompt for every assistant or design a different one for each role? Explain your
reasoning.**

I would use different system prompts for each assistant. The Customer Support assistant should focus on helping customers with bookings, baggage and general questions, while the Flight Operations assistant should focus more on scheduling and operations. The Aircraft Maintenance assistant should give more technical answers about aircraft and safety. Different system prompts can make each assistant more focused on its own role and give more relevant answers.

---

**3.Which of the experiments in Part 3 had the greatest influence on the quality of the
responses? Support your answer with observations from your own experiments.**

I think the additional context experiment had the biggest effect on the quality of the responses. Without the AeroWing reference information, Gemini could only give a general answer about the A320 and baggage. After I provided the AeroWing information, it gave a more specific answer about Wi-Fi being available on selected routes and the exact baggage allowance. This showed me that giving the model relevant information can make its answers more accurate and useful.


(your answers here)